# 02 做多做空与结构化输出

## 本课学习目标

- A. 量化金融主线：做多（Long）和做空（Short）
零基础解释：做多希望上涨获利，做空希望下跌获利但风险可能更大。
- B. 大语言模型主线：结构化输出（Structured Output）和 JavaScript 对象表示法（JavaScript Object Notation，JSON）
零基础解释：固定字段能让程序可靠读取模型结果。
- C. 两条线如何连接：把市场数据和新闻文本转成可检查的表格信号。
- D. 可运行实验：用数字案例计算多空收益，并演示 Pydantic 校验合法和非法情绪结果。
- E. 结果解释：观察表格、图表和结构化输出。
- F. 常见错误：把回测收益当成未来收益、把 Mock 当成真实模型。
- G. 课后练习：修改一个参数并重新运行。
- H. 本课术语表：见本课各小节。

## 本课最终输出

一个离线实验输出，不联网、不调用真实模型、不产生真实订单。

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "learning").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "learning" / "data"


## 开仓和平仓

### 开仓（Open a Position）

零基础解释：开仓是建立一笔新的交易持仓。交易前持仓为 0，交易后有了正或负的持仓。

### 平仓（Close a Position）

零基础解释：平仓是结束之前建立的持仓。平仓后持仓回到 0，本次交易周期结束。

### 四个数字案例

以下用 1 股为最小单位，说明四种基本操作。假设初始资金充足，忽略交易费用。

#### 案例 1：买入开仓（做多）
- 以 100 元买入 1 股。
- 持仓从 **0 股 → 1 股**。
- 这是做多开仓。

#### 案例 2：卖出平仓（做多了结）
- 之前持有 1 股（成本 100 元），以 120 元卖出。
- 持仓从 **1 股 → 0 股**。
- 利润：**120 − 100 = 20 元**。

#### 案例 3：卖出开仓（做空）
- 以 100 元借入并卖出 1 股，建立空头持仓。
- 持仓可以表示为 **−1 股**。
- 这是做空开仓。

#### 案例 4：买入平仓（做空了结）
- 之前空头 1 股（卖空价 100 元），股价跌至 70 元时买回 1 股归还。
- 持仓从 **−1 股 → 0 股**。
- 利润：**100 − 70 = 30 元**。

### 持仓变化总结

| 原持仓 | 操作 | 新持仓 | 含义 |
|--------|------|--------|------|
| 0 | 买入 1 股 | 1 | 做多开仓 |
| 1 | 卖出 1 股 | 0 | 做多平仓 |
| 0 | 卖出 1 股 | −1 | 做空开仓 |
| −1 | 买入 1 股 | 0 | 做空平仓 |

**关键理解："卖出"不一定是"做空"。**
- 卖出已有的股票是平仓（平多）；
- 没有持仓却借入并卖出，才是股票做空（开空）。

下面用 Python 模拟持仓状态变化。

In [ ]:
# 持仓状态变化实验
print("=" * 40)
print("做多流程：0 → 1 → 0")
position = 0
print(f"初始持仓：{position}")
position += 1  # 买入开仓
print(f"买入开仓后持仓：{position}")
position -= 1  # 卖出平仓
print(f"卖出平仓后持仓：{position}")
print()
print("做空流程：0 → -1 → 0")
position = 0
print(f"初始持仓：{position}")
position -= 1  # 卖出开仓（做空）
print(f"卖出开仓后持仓：{position}")
position += 1  # 买入平仓
print(f"买入平仓后持仓：{position}")
print("=" * 40)
print('注意：做空的卖出开仓和做多的卖出平仓，')
print('虽然操作都是卖出，但含义完全不同。')

## 可运行实验

下面代码只读取 `learning/data` 下的合成数据。

In [ ]:
from datetime import datetime
import pandas as pd
from pydantic import ValidationError
from learning.src.schemas import SentimentResult
long_profit = 120 - 100
short_profit = 100 - 70
display(pd.DataFrame({"case": ["long", "short"], "profit": [long_profit, short_profit]}))
ok = SentimentResult(label="positive", score=0.8, confidence=0.9, reason="字段合法", ticker="AAA", published_at=datetime.now(), provider="offline", model_name="demo")
display(ok.model_dump())
try:
    SentimentResult(label="great", score=2, confidence=1.5, reason="", ticker="", published_at=datetime.now(), provider="offline", model_name="demo")
except ValidationError as exc:
    print(exc)

## 结尾总结

你现在应该理解：量化交易的基本操作（做多/做空/开仓/平仓）和结构化输出（JSON/Pydantic）是连接交易逻辑与程序验证的关键。

本课核心收获：
- 做多希望上涨获利，做空希望下跌获利；
- 开仓建立持仓，平仓结束持仓；
- "卖出"不一定是"做空"——卖出已有股票是平多，借入卖出才是做空；
- 结构化输出使用固定字段，Pydantic 可以自动校验；
- 非法字段（如 label="great"）会被 Pydantic 直接拒绝。

哪些结果不能解释为策略一定赚钱：任何图表和收益数字都只是合成数据上的教学结果。

本课使用了哪些英文专业词：
- Long（做多）
- Short（做空）
- Open a Position（开仓）
- Close a Position（平仓）
- Structured Output（结构化输出）
- JSON（JavaScript 对象表示法）
- Pydantic（Python 数据校验库）

### 常见错误

1. **把卖出持仓误认为做空**：卖出自己持有的股票是平多，做空需要先借入股票再卖出。
2. **忘记做空需要后续买回**：做空不是"卖掉就结束了"，必须在未来某个时间买回归还。
3. **把非法 JSON 当作结构化输出**：字段名拼错、类型不对、缺少必填字段，都会导致程序解析失败。Pydantic 可以在接收时就拦截这些错误。
4. **混淆开仓方向**：买入开仓是做多，卖出开仓是做空，两者方向相反。

### 课后练习

1. **模拟完整交易**：写一段 Python 代码，从 0 持仓开始，做多开仓（买入），价格上涨后平仓，计算盈亏。
2. **模拟做空交易**：写一段 Python 代码，模拟做空开仓（卖出），价格下跌后买回平仓，计算盈亏。
3. **校验实验**：故意构造一个 label 不是 "positive"/"neutral"/"negative" 中任何一个的 SentimentResult，观察 Pydantic 是否拒绝。

下一课与本课有什么关系：下一课会在结构化输出的基础上，引入收益率、波动率和最大回撤等风险指标，并讲解 Mock Provider 的情绪分类。